# Exercise 7: From the log to the lake

Every run of your job does three things: **read** the new records of the topic `logs`,
**write** them into the lake (Parquet files in `data/lake/`), **remember** where it stopped.

This notebook gives you the building blocks (`tools/lakelib.py`). You write one function,
`run_once()`. Save your final version of this notebook into your `solution/` folder.

In [1]:
import sys; sys.path.append("/home/jovyan/tools")
from lakelib import *

(installing confluent-kafka==2.11.0 ...)
(installing duckdb==1.5.6 ...)


## 1. The example: a job that copies everything

It reads the topic from the beginning, every time. Run the next two cells and look at the
checker's last line.

In [2]:
def run_once():
    new = read_new(None)          # None = from the beginning of the topic
    write_files(new, LAKE)        # straight into the lake

reset()                           # empty lake
run_once()
check()

reset: lake, staging and state deleted
read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-023331-ce9369

Lake check — topic 'logs' (4 partitions)   2026-11-25 02:37:12
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 493,378)     493,378           0          0           0
        1           [0, 461,936)     461,936           0          0           0
        2           [0, 522,640)     522,640           0          0           0
        3           [0, 522,046)     522,046           0          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   2,000,000           0          0           0

files under /home/jovyan/data/lake: 4 data files, 111.7 MB, 27.93 MB pe

{'topic': 2000000,
 'lake': 2000000,
 'dup': 0,
 'miss': 0,
 'lag': 0,
 'files': 4,
 'mb': 111.707733,
 'ok': True,
 'problems': []}

In [3]:
run_once()                        # the next run
check()

read_new: 2,000,000 new records
write_files: 2,000,000 records -> /home/jovyan/data/lake/logs/run-20261125-023712-60f207

Lake check — topic 'logs' (4 partitions)   2026-11-25 02:39:18
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 493,378)     986,756     493,378          0           0
        1           [0, 461,936)     923,872     461,936          0           0
        2           [0, 522,640)   1,045,280     522,640          0           0
        3           [0, 522,046)   1,044,092     522,046          0           0
-------------------------------------------------------------------------------
    total      2,000,000 records   4,000,000   2,000,000          0           0

files under /home/jovyan/data/lake: 8 data files, 223.4 MB, 27.93 MB per file on average
check took 3.3 s

Exa

{'topic': 2000000,
 'lake': 4000000,
 'dup': 2000000,
 'miss': 0,
 'lag': 0,
 'files': 8,
 'mb': 223.415466,
 'ok': False,
 'problems': ['2,000,000 duplicates']}

## Why does the checker fail?
After the first run the check passes, every record is in the lake once.  
But the example never saves where it stopped, read_new(None) always starts from the beginning of the topic.  
So the second run copies all the records again, into a new run folder.  
Now every record is in the lake twice, the checker counts 2 million duplicates and fails.  
Nothing is missing, the job is just missing the remember step.

## 2. The building blocks

| Block | What it does |
|---|---|
| `load_state()` | where the last run stopped, from a small state file (`None` if there is none) |
| `save_state(new.end)` | stores where this run stopped in the state file |
| `lake_position()` | where the last run stopped, read from the lake itself (`None` if the lake is empty) |
| `read_new(start)` | the records after `start`, up to the end of the topic *now* (`new.end`) |
| `write_files(new, folder)` | writes them as Parquet into a new run folder inside `LAKE` or `STAGING` |
| `publish(run)` | moves a finished run folder from `STAGING` into the lake: one rename, a single step |

`LAKE` is what readers see; `STAGING` is a folder next to it that nobody reads.
`new.df.show()` shows the records of a run as a Spark DataFrame.

## 3. Your architecture

Write `run_once()` for the architecture you chose (A, B or C, see the README).
The order of the calls *is* your architecture.

## My choice: B, the lake is the truth
I went with B, because the job doesn't need a separate place to store where it stopped.  
It looks at the lake and takes the highest offset per partition, so the position and the data can't get out of sync.  
Every run is written into the staging folder first, which readers never look at.  
If a run stops halfway, the leftovers are only in staging, and the next run clears them anyway.  
When the run is done, publish moves the folder into the lake with one rename, so the whole run shows up at once.  
That's why there is nothing to save at the end, the lake already tells the next run where to continue.

In [4]:
def run_once():
    # 1. where did the last run stop? (from the lake itself)
    start = lake_position()
    # 2. read what is new
    new = read_new(start)
    # 3. write it to staging first, then move it into the lake in one step
    run = write_files(new, STAGING)
    publish(run)
    # 4. nothing to save, the lake already knows where we stopped

## 4. Check it

`check_all` empties the lake, then runs your job in five situations: a first run, new records,
nothing new, a run that stops **while** writing, a run that stops **after** writing.
It takes 2-4 minutes.

Architecture C: readers use a view, so check through it: `check_all(run_once, lake_sql=DEDUP_SQL)`.

In [5]:
check_all(run_once)

reset: lake, staging and state deleted



situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,000,000           0        0           0 122.1s  PASS
2 new records arrive, run again         2,100,000           0        0           0  39.3s  PASS
3 nothing new, run again                2,100,000           0        0           0   1.8s  PASS
4 run stops WHILE writing: right after  2,100,000           0        0     100,000         PASS
  ... then the next run                 2,200,000           0        0           0  31.4s  PASS
5 run stops AFTER writing: right after  2,200,000           0        0     100,000         PASS
  ... then the next run                 2,300,000           0        0           0  32.5s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: YES


[('1 first run',
  {'topic': 2000000,
   'lake': 2000000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 111.707733,
   'ok': True,
   'problems': []},
  122.10656380653381),
 ('2 new records arrive, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.867547,
   'ok': True,
   'problems': []},
  39.30807685852051),
 ('3 nothing new, run again',
  {'topic': 2100000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 112.867547,
   'ok': True,
   'problems': []},
  1.7893593311309814),
 ('4 run stops WHILE writing: right after',
  {'topic': 2200000,
   'lake': 2100000,
   'dup': 0,
   'miss': 0,
   'lag': 100000,
   'files': 8,
   'mb': 112.867547,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2200000,
   'lake': 2200000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 12,
   'mb': 114.025791,
   'ok': True,
   'problems': []},


## 5. Extra: why not A or C?
To compare, I also ran A and C through the same five situations.  
In A the job writes straight into the lake, and saves its position in the state file after writing.  
If it stops while writing or after writing, some records are already in the lake, but the state file still has the old position, so the next run copies them again.

In [6]:
def run_once_A():
    start = load_state()    # where the last run stopped, from the state file
    new = read_new(start)
    write_files(new, LAKE)  # straight into the lake
    save_state(new.end)     # save the position after writing

check_all(run_once_A)

reset: lake, staging and state deleted



situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,300,000           0        0           0 127.2s  PASS
2 new records arrive, run again         2,400,000           0        0           0  36.2s  PASS
3 nothing new, run again                2,400,000           0        0           0   0.1s  PASS
4 run stops WHILE writing: right after  2,440,474           0        0      59,526         PASS
  ... then the next run                 2,540,474      40,474        0           0  35.9s  FAIL
5 run stops AFTER writing: right after  2,640,474      40,474        0           0         FAIL
  ... then the next run                 2,740,474     140,474        0           0  36.1s  FAIL
-----------------------------------------------------------------------------------------------
Exactly once after every run: NO — see

[('1 first run',
  {'topic': 2300000,
   'lake': 2300000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 116.880601,
   'ok': True,
   'problems': []},
  127.2037000656128),
 ('2 new records arrive, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 118.039277,
   'ok': True,
   'problems': []},
  36.24579644203186),
 ('3 nothing new, run again',
  {'topic': 2400000,
   'lake': 2400000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 118.039277,
   'ok': True,
   'problems': []},
  0.13375616073608398),
 ('4 run stops WHILE writing: right after',
  {'topic': 2500000,
   'lake': 2440474,
   'dup': 0,
   'miss': 0,
   'lag': 59526,
   'files': 9,
   'mb': 118.474389,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2500000,
   'lake': 2540474,
   'dup': 40474,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 119.621183,
   'ok': False,
   'problems': [

C is the same job as A.  
The difference is that readers go through a view, that keeps every record only once (DEDUP_SQL).

In [7]:
def run_once_C():
    start = load_state()
    new = read_new(start)
    write_files(new, LAKE)
    save_state(new.end)

check_all(run_once_C, lake_sql=DEDUP_SQL)

reset: lake, staging and state deleted



situation                                 in lake  duplicates  missing  not loaded    run  check
-----------------------------------------------------------------------------------------------
1 first run                             2,600,000           0        0           0 115.5s  PASS
2 new records arrive, run again         2,700,000           0        0           0  34.5s  PASS
3 nothing new, run again                2,700,000           0        0           0   0.2s  PASS
4 run stops WHILE writing: right after  2,725,196           0        0      74,804         PASS
  ... then the next run                 2,800,000           0        0           0  31.1s  PASS
5 run stops AFTER writing: right after  2,900,000           0        0           0         PASS
  ... then the next run                 2,900,000           0        0           0  33.1s  PASS
-----------------------------------------------------------------------------------------------
Exactly once after every run: YES


[('1 first run',
  {'topic': 2600000,
   'lake': 2600000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 4,
   'mb': 122.049195,
   'ok': True,
   'problems': []},
  115.46678400039673),
 ('2 new records arrive, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 123.207921,
   'ok': True,
   'problems': []},
  34.49982976913452),
 ('3 nothing new, run again',
  {'topic': 2700000,
   'lake': 2700000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 8,
   'mb': 123.207921,
   'ok': True,
   'problems': []},
  0.183135986328125),
 ('4 run stops WHILE writing: right after',
  {'topic': 2800000,
   'lake': 2725196,
   'dup': 0,
   'miss': 0,
   'lag': 74804,
   'files': 9,
   'mb': 123.499982,
   'ok': True,
   'problems': []},
  None),
 ('  ... then the next run',
  {'topic': 2800000,
   'lake': 2800000,
   'dup': 0,
   'miss': 0,
   'lag': 0,
   'files': 13,
   'mb': 124.65295,
   'ok': True,
   'problems': []},
  3

C passes, but only because of the view.  
The duplicates are still in the files, the check below reads the lake without the view.  
So every reader has to use the view, and removing the duplicates on every query makes reading slower.

In [8]:
check()   # the lake without the view


Lake check — topic 'logs' (4 partitions)   2026-11-25 02:54:46
lake: read_parquet('/home/jovyan/data/lake/logs/**/*.parquet')

partition          topic offsets     in lake  duplicates    missing  not loaded
-------------------------------------------------------------------------------
        0           [0, 758,661)     810,377      51,716          0           0
        1           [0, 708,901)     735,797      26,896          0           0
        2           [0, 691,676)     719,030      27,354          0           0
        3           [0, 740,762)     759,992      19,230          0           0
-------------------------------------------------------------------------------
    total      2,900,000 records   3,025,196     125,196          0           0

files under /home/jovyan/data/lake: 21 data files, 127.0 MB, 6.05 MB per file on average
check took 4.0 s

Exactly once: FAIL — 125,196 duplicates


{'topic': 2900000,
 'lake': 3025196,
 'dup': 125196,
 'miss': 0,
 'lag': 0,
 'files': 21,
 'mb': 126.971514,
 'ok': False,
 'problems': ['125,196 duplicates']}